# Energy Model — Household Energy Footprint

This notebook computes a household energy footprint for a chosen country (default Canada) by combining:

1. **FIGARO** multi-regional input–output tables (Z, Y, X) and metadata.
2. **IEA World Energy Balances** total final consumption (TJ) per country × IEA sector.
3. **NACE↔IEA** bridge matrix to distribute IEA energy into NACE sectors.
4. **SHS** (Statistics Canada Survey of Household Spending) weighted expenditures per COICOP category.
5. **NACE↔COICOP** bridge matrix + **RAS balancing** to redistribute IO household consumption into COICOP categories.

Outputs (per COICOP category):
- `e_COICOP_indir` — indirect energy intensity (TJ per million EUR)
- `e_COICOP_dir` — direct energy intensity (road + residential)
- `e_COICOP_TJperMlnEur` — combined two-column table (TJ per million EUR)

Run cells in order with **Shift + Enter**.

## Setup — imports, file paths, country selection

Pulls in the four project modules:

| Module | Function | Purpose |
|---|---|---|
| `Figaro_economic_data` | `figaro_economic_data()` | Reads FIGARO matrix CSVs + `meta.xlsx`, builds X/Y/Z. |
| `SHS_EDM_2019_data` | `shs_2019_data()` | Loads SHS `.sas7bdat`, multiplies expenditures by household weight. |
| `NACE_COICOP_mask` | `nace_to_coicop_mask()` | Loads the NACE→COICOP bridge mask. |
| `RAS_balancing` | `ras_balancing()` | Distributes household IO consumption across COICOP categories using RAS. |

Defines:
- File paths (`DATA_DIR`, `FIGARO_FOLDER`, `SAS_PATH`, `BRIDGE_PATH`, `IEA_PATH`, `NACE_IEA_PATH`)
- Country / category to analyse: `COUNTRY_CODE`, `HH_CATEGORY`, `FD_CATEGORIES`

In [1]:
import os
import pandas as pd

from Figaro_economic_data import figaro_economic_data
from SHS_EDM_2019_data import shs_2019_data
from NACE_COICOP_mask import nace_to_coicop_mask
from RAS_balancing import ras_balancing

# --- File paths (change once per machine) ---
DATA_DIR = r"C:\Users\marta.baltruszewicz\OneDrive - Asplan Viak\Documents\additional_projects\Canada_hh_energy\Data"
FIGARO_FOLDER = r"C:\Users\marta.baltruszewicz\OneDrive - Asplan Viak\Documents - 403001-01 Utvikling og vedlikehold klimakost\General\Utslippsfaktorer\Kryssløpsanalyse\FIGARO data"

SAS_PATH = os.path.join(
    DATA_DIR,
    r"OneDrive_2026-03-09\Survey of Household Spending\SHS_EDM_2019\Data\SAS\pumf_shs2019.sas7bdat",
)
BRIDGE_PATH = os.path.join(DATA_DIR, "COICOP_to_FIGARO_Bridge_2019.xlsx")
IEA_PATH = os.path.join(DATA_DIR, "IEA_data_2019.xlsx")
NACE_IEA_PATH = os.path.join(DATA_DIR, "IEA_NACE_mapping.xlsx")

# --- Country / household-category to analyse ---
COUNTRY_CODE = "CA"          # country code in regions.iloc[:, 1]
HH_CATEGORY = "P3_S14"       # household consumption final-demand category
FD_CATEGORIES = ["P3_S13", "P3_S14", "P3_S15", "P51G", "P5M"]

## Cell 1 — Load FIGARO economic input–output data

Calls **`figaro_economic_data(...)`** which:

1. Lists matrix CSVs (`matrix_eu-ic-io_ind-by-ind_2501ed_<year>.csv`) under `FIGARO_FOLDER` for the requested year range.
2. Loads each CSV into `pd.read_csv` and assembles the three IO matrices:
   - `Z` — intermediate flows, shape `(m*n, m*n, n_years)` (= 2852 × 2852 × 1 for 2019)
   - `Y` — final demand, shape `(m*n, m*k, n_years)` (= 2852 × 230)
   - `X` — total output, shape `(m*n, 1, n_years)`
3. Loads `meta.xlsx` sheets `country`, `sector`, `final_demand` into `meta` (a `SimpleNamespace`).
4. Extracts the household final-demand vector for `COUNTRY_CODE` and `HH_CATEGORY` and returns it under the key `f"{COUNTRY_CODE}_Y_hh"`.

Variables created here:
- `X`, `Y`, `Z`, `meta`, `regions`, `sectors`, `FDcategories`
- `Y_hh` — the household FD vector (length 2852) for the chosen country

In [2]:
result = figaro_economic_data(
    figaro_folder=FIGARO_FOLDER,
    first_year=2019,
    last_year=2019,
    edition_economic_data=2501,
    country_code=COUNTRY_CODE,
    hh_category=HH_CATEGORY,
    fd_categories=FD_CATEGORIES,
)
X = result["X"]
Y = result["Y"]
Z = result["Z"]
meta = result["meta"]

regions = meta.regions
sectors = meta.sectors
FDcategories = meta.FDcategories

Y_hh = result[f"{COUNTRY_CODE}_Y_hh"]

Found 14 matching files:
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2010.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2011.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2012.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2013.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2014.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2015.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2016.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2017.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2018.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2019.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2020.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2021.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2022.csv
  indxind202501ed\matrix_eu-ic-io_ind-by-ind_2501ed_2023.csv
Loading matrix_eu-ic-io_ind-by-ind_2501ed_2019.csv ...

Loaded 1 years: [2019]
  Importing FIGARO economic data for year:

## Cell 2 — Load IEA energy data

Two `pd.read_excel(...)` reads:

1. **`IEA_FC_tot`** — from `IEA_data_2019.xlsx`, tab `IEA_to_Figaro_data`, columns A:F and M.
   - One row per (country, IEA sector) pair; the value column is TJ.
   - Includes road indirect and international bunkers.
2. **`NACE_to_IEA_mask`** — from `IEA_NACE_mapping.xlsx`, tab `NACE_IEA_Bridge`, columns A:B + D:Z, rows 1–62.
   - Many-to-many bridge: rows = 62 NACE sectors, columns 2+ = 23 IEA sectors. Values are 0/1.

In [3]:
IEA_FC_tot = pd.read_excel(
    IEA_PATH,
    sheet_name="IEA_to_Figaro_data",
    usecols="A:F,M",
)

NACE_to_IEA_mask = pd.read_excel(
    NACE_IEA_PATH,
    sheet_name="NACE_IEA_Bridge",
    usecols="A:B,D:Z",
    header=0,
    nrows=62,
)

print(f"IEA_FC_tot shape:       {IEA_FC_tot.shape}")
print(f"NACE_to_IEA_mask shape: {NACE_to_IEA_mask.shape}")

IEA_FC_tot shape:       (1058, 7)
NACE_to_IEA_mask shape: (62, 25)


## Cell 3 — Indirect energy intensity + household indirect footprint

Calls **`indir_E_e(...)`** from `Indir_E_e.py`. It performs the seven-step indirect-energy pipeline:

1. Sums rows of `Z` corresponding to the three energy-supplying NACE sectors (`B`, `C19`, `D35`) across all countries → `sum_Z_sup_E`.
2. Distributes that supply across the 23 IEA sectors per country using `NACE_to_IEA_mask`.
3. Row-normalises so per-country IEA shares sum to 1 — `shares_sup_IEA_NACE`.
4. Multiplies the shares by country × IEA totals from `IEA_FC_tot` → **`E_TJ`** (energy extension in TJ over the 2852-country-sector grid).
5. Element-wise division: **`e_NACE = E_TJ / X`** — energy intensity (TJ per million EUR).
6. Builds Leontief inverse **`L = (I − A)⁻¹`** where `A = Z * diag(1/X)`.
7. Household indirect footprint: **`hh_EF_NACE_indir = e_NACE * (L @ Y_hh)`**.

Outputs: `E_TJ`, `e_NACE`, `L`, `hh_EF_NACE_indir` (all length-2852 vectors / 2852×2852 for L).

In [4]:
import numpy as np
from Indir_E_e import indir_E_e

m = result["m"]   # 46 regions
n = result["n"]   # 62 sectors

indir = indir_E_e(
    Z=Z, X=X, Y_hh=Y_hh,
    sectors=sectors,
    NACE_to_IEA_mask=NACE_to_IEA_mask,
    IEA_FC_tot=IEA_FC_tot,
    m=m, n=n,
)
E_TJ = indir["E_TJ"]
e_NACE = indir["e_NACE"]
L = indir["L"]
hh_EF_NACE_indir = indir["hh_EF_NACE_indir"]

E_TJ vs IEA_FC_tot total: 256,114,094.57 vs 256,114,094.57 (|diff|=0.0000)
hh_EF_NACE_indir shape: (2852,), total = 2,905,829.77 TJ


## Cell 4 — Load SHS 2019 household spends

Calls **`shs_2019_data(sas_path, exchange_rate)`** from `SHS_EDM_2019_data.py`. The loader:

- Reads the SAS file with `pd.read_sas`.
- Multiplies each expenditure column by `WeightD` (household weight) so each row represents a population-scaled spend.
- Returns a **dict with two flavours** of the same weighted data:
  - **`CAD_WGHT`** → `SHS_2019_data_CAD_WGHT` — values in CAD per household, weighted by WeightD.
  - **`Mln_EUR_WGHT`** → `SHS_2019_data_Mln_EUR_WGHT` — the same values converted to million EUR (CAD ÷ 1.49 ÷ 1 000 000).

Each row is one survey household; columns are `CaseID`, `WeightD`, then the 75 weighted expenditure columns (named like `FD003_wght`, …).

After loading we **strip the `_wght` suffix** from both DataFrames so downstream tables use clean COICOP codes (`FD003`, `SH003`, …). The variable-name suffix `_WGHT` still marks the data as weighted. `COICOP_code` is saved as the canonical 75-code list for later matching.

In [5]:
shs_dict = shs_2019_data(sas_path=SAS_PATH, exchange_rate=1.49)
SHS_2019_data_CAD_WGHT     = shs_dict["CAD_WGHT"]
SHS_2019_data_Mln_EUR_WGHT = shs_dict["Mln_EUR_WGHT"]

# Strip the "_wght" suffix so we have clean COICOP codes.
SHS_2019_data_CAD_WGHT.columns     = [c.removesuffix("_wght") for c in SHS_2019_data_CAD_WGHT.columns]
SHS_2019_data_Mln_EUR_WGHT.columns = [c.removesuffix("_wght") for c in SHS_2019_data_Mln_EUR_WGHT.columns]

COICOP_code = SHS_2019_data_CAD_WGHT.columns[2:].tolist()   # skip CaseID + WeightD
print(f"COICOP_code ({len(COICOP_code)} categories): first 5 = {COICOP_code[:5]}")

Imported SHS_2019_data in 0.36 seconds (exchange rate 1.49 CAD/EUR)
COICOP_code (75 categories): first 5 = ['FD003', 'FD990', 'SH003', 'SH011', 'SH082']


## Cell 5 — NACE-to-COICOP bridge mask

Calls **`nace_to_coicop_mask(...)`** from `NACE_COICOP_mask.py`:

- Reads tab `COICOP_to_FIGARO_Bridge` in `COICOP_to_FIGARO_Bridge_2019.xlsx`, range B8:BZ70 (62 × 77).
- Layout: column 0 = NACE code, column 1 = NACE name, columns 2+ = 0/1 mapping to 75 COICOP categories.
- Validates against `sectors` (row order) and `SHS_2019_data_CAD_WGHT` (column-name match — values not used).

In [6]:
NACE_to_COICOP_mask = nace_to_coicop_mask(
    bridge_path=BRIDGE_PATH,
    sectors=sectors,
    shs_data=SHS_2019_data_CAD_WGHT,
)

NACE_to_COICOP_mask shape: (62, 77)
OK: first column matches sectors order (62 sectors)
OK: mask column names match SHS columns (75 categories)


## Cell 6 — RAS balancing + COICOP distribution of household energy

Five steps:

1. **`SHS_2019_75cat`** — column sums of the 75 expenditure columns from `SHS_2019_data_Mln_EUR_WGHT` (million EUR).
2. **`ras_balancing(...)`** from `RAS_balancing.py` — redistributes household IO consumption (`Y_hh` aggregated per NACE) across COICOP categories using `NACE_to_COICOP_mask` as a 0/1 structural mask. Mode `"rows"` enforces row sums exactly; column sums are best-effort.
   - The CAD→EUR conversion is no longer done inside RAS — it lives in the SHS loader. We pass `SHS_2019_data_Mln_EUR_WGHT` directly via `shs_eur=...`.
   - Internally: distributes the diff between IO and SHS totals across COICOP shares, aggregates `Y_hh` across countries to a 62-vector, runs the RAS scaling.
   - Returns: **`RAS_NACE_COICOP_spends`** (62 × 75, in million EUR) and **`RAS_nace_to_coicop_shares`** (each row sums to 1).
3. **`hh_EF_NACE`** — aggregate `hh_EF_NACE_indir` by summing across all 46 countries for each of the 62 sectors.
4. **`EF_indir_COICOP`** — multiply each row of the shares matrix by `hh_EF_NACE[i]` to spread the per-sector TJ across COICOP categories.
5. **`e_COICOP_indir`** — column sums of `EF_indir_COICOP` divided by column sums of `RAS_NACE_COICOP_spends` → indirect energy intensity (TJ / million EUR) per COICOP.

In [7]:
SHS_2019_75cat = SHS_2019_data_Mln_EUR_WGHT.iloc[:, 2:].sum().to_numpy(dtype=float)
print(f"SHS_2019_75cat shape: {SHS_2019_75cat.shape}, sum (mln EUR) = {SHS_2019_75cat.sum():,.2f}")

ras_result = ras_balancing(
    nace_coicop_mask=NACE_to_COICOP_mask,
    shs_eur=SHS_2019_data_Mln_EUR_WGHT,
    Y=Y,
    regions=regions,
    m=result["m"],
    n=result["n"],
    k=result["k"],
    year_idx=0,
    country_code=COUNTRY_CODE,
    fd_categories=FD_CATEGORIES,
    hh_category=HH_CATEGORY,
    mode="rows",
)
RAS_NACE_COICOP_spends = ras_result["balanced"]   # units: million EUR
RAS_nace_to_coicop_shares = ras_result["RAS_nace_to_coicop_shares"]
print(f"\nBalanced matrix shape: {RAS_NACE_COICOP_spends.shape}")
print(f"Converged: {ras_result['converged']}, iterations: {ras_result['iterations']}")

# Aggregate household energy footprint to 62 NACE sectors
n_sec = result["n"]
hh_EF_NACE = pd.Series(
    np.array([hh_EF_NACE_indir[s::n_sec].sum() for s in range(n_sec)]),
    index=sectors.iloc[:, 1].astype(str).str.strip().tolist(),
    name="hh_EF_NACE",
)
print(f"hh_EF_NACE shape: {hh_EF_NACE.shape}, total = {hh_EF_NACE.sum():,.2f} TJ")

# Distribute per-NACE energy footprint across COICOP categories
EF_indir_COICOP = RAS_nace_to_coicop_shares.multiply(hh_EF_NACE, axis=0)
print(f"EF_indir_COICOP shape: {EF_indir_COICOP.shape}, total = {EF_indir_COICOP.values.sum():,.2f} TJ")

# Indirect energy intensity per COICOP category
e_COICOP_indir = EF_indir_COICOP.sum(axis=0) / RAS_NACE_COICOP_spends.sum(axis=0)
e_COICOP_indir.name = "e_COICOP_indir"
print(f"e_COICOP_indir shape: {e_COICOP_indir.shape} (units: TJ / million EUR)")

SHS_2019_75cat shape: (75,), sum (mln EUR) = 675,164.01
CA_Y_hh sum (EUR):       800,246.45
SHS sum (EUR, original): 675,164.01
Difference distributed:  125,082.44
SHS sum (EUR, adjusted): 800,246.45
Rows-only mode: row err 2.91e-11, col err 1.24e+04 (columns are best-effort)

RAS_nace_to_coicop_shares row-sum check: max deviation = 2.22e-16
  OK: every non-empty row sums to 100%.

Balanced matrix shape: (62, 75)
Converged: True, iterations: 1
hh_EF_NACE shape: (62,), total = 2,905,829.77 TJ
EF_indir_COICOP shape: (62, 75), total = 2,905,829.77 TJ
e_COICOP_indir shape: (75,) (units: TJ / million EUR)


## Cell 7 — Direct energy intensity per COICOP

Reads **`dir_E_road_resid`** from `IEA_data_2019.xlsx`, tab `Resid_Road_split_to_COICOP`, range A28:C33.

Layout: column A = description, column B = COICOP code, column C = direct energy value in TJ.

For each row in `dir_E_road_resid`:
- Match the COICOP code against `RAS_NACE_COICOP_spends.columns`.
- Compute `e_COICOP_dir[code] = direct_energy_value / sum(RAS_NACE_COICOP_spends[code])`.

Result `e_COICOP_dir` — a Series indexed by COICOP code with intensity in TJ / million EUR for the road/residential categories only.

In [8]:
dir_E_road_resid = pd.read_excel(
    IEA_PATH,
    sheet_name="Resid_Road_split_to_COICOP",
    usecols="A:C",
    header=0,
    skiprows=27,
    nrows=5,
)
print(f"dir_E_road_resid shape: {dir_E_road_resid.shape}")
print(dir_E_road_resid)

e_COICOP_dir = pd.Series(dtype=float, name="e_COICOP_dir")
for _, row in dir_E_road_resid.iterrows():
    code = str(row.iloc[1]).strip()
    val = float(row.iloc[2])
    if code not in RAS_NACE_COICOP_spends.columns:
        raise ValueError(f"COICOP code {code!r} not found in RAS_NACE_COICOP_spends columns")
    col_sum = RAS_NACE_COICOP_spends[code].sum()
    e_COICOP_dir[code] = val / col_sum

print(f"\ne_COICOP_dir ({len(e_COICOP_dir)} categories):")
print(e_COICOP_dir)

dir_E_road_resid shape: (5, 3)
                                     COICOP category COICOP CODE  \
0                                 SH034 - Other fuel       SH034   
1                                SH033 - Natural gas       SH033   
2                                SH032 - Electricity       SH032   
3  SH062 - Electricity and fuel (e.g. natural gas...       SH062   
4  TR036 - Gas and other fuels (all vehicles and ...       TR036   

   Direct energy (TJ)  
0       153861.332900  
1       621028.801800  
2       596755.090733  
3        28305.708367  
4       881865.181600  

e_COICOP_dir (5 categories):
SH034    286.002517
SH033    101.753334
SH032     54.816135
SH062     42.269489
TR036     34.974062
Name: e_COICOP_dir, dtype: float64


## Cell 8 — Combine indirect + direct COICOP intensities

Build the final two-column table **`e_COICOP_TJperMlnEur`** (75 rows × 2 columns):

- `indir` — `e_COICOP_indir` (every COICOP category has a value).
- `dir` — `e_COICOP_dir` aligned to the same COICOP index; categories not in `e_COICOP_dir` (i.e. not road/residential) get **0**.

Both columns are in TJ per million EUR. Total energy intensity for a category is `indir + dir`.

In [9]:
e_COICOP_TJperMlnEur = pd.DataFrame({
    "indir": e_COICOP_indir,
    "dir":   e_COICOP_dir.reindex(e_COICOP_indir.index, fill_value=0.0),
})
print(f"e_COICOP_TJperMlnEur shape: {e_COICOP_TJperMlnEur.shape}")
print(e_COICOP_TJperMlnEur.head())

# Energy footprint (TJ) per COICOP — same logic but with absolute TJ values.
#   EF_indir_COICOP = column sums of EF_indir_COICOP
#   EF_dir_COICOP   = direct TJ values from dir_E_road_resid, matched to COICOP;
#                     categories not in dir_E_road_resid get 0.
EF_dir_COICOP = pd.Series(
    {str(row.iloc[1]).strip(): float(row.iloc[2]) for _, row in dir_E_road_resid.iterrows()},
    name="EF_dir_COICOP",
)
EF_COICOP_TJ_indir_dir = pd.DataFrame({
    "EF_indir_COICOP": EF_indir_COICOP.sum(axis=0),
    "EF_dir_COICOP":   EF_dir_COICOP.reindex(EF_indir_COICOP.columns, fill_value=0.0),
})
print(f"\nEF_COICOP_TJ_indir_dir shape: {EF_COICOP_TJ_indir_dir.shape} (units: TJ)")
EF_COICOP_TJ_indir_dir.head()

e_COICOP_TJperMlnEur shape: (75, 2)
          indir  dir
FD003  3.058335  0.0
FD990  1.297076  0.0
SH003  0.803892  0.0
SH011  0.803892  0.0
SH082  9.078532  0.0

EF_COICOP_TJ_indir_dir shape: (75, 2) (units: TJ)


,EF_indir_COICOP,EF_dir_COICOP
FD003,269558.460289,0.0
FD990,47745.796562,0.0
SH003,39786.823308,0.0
SH011,61191.248638,0.0
SH082,9893.875542,0.0
